# Credits

This is heavily influenced or copied from https://github.com/pytorch/tutorials

# Autograd: automatic differentiation自动微分

Central to all neural networks in PyTorch is the ``autograd`` package.
Let’s first briefly visit this, and we will then go to training our first neural network.

The `autograd` package provides automatic differentiation for all operations on Tensors.
It is a define-by-run framework, which means that your backprop is defined by how your code is run, and that every single iteration can be different.

Let us see this in more simple terms with some examples.

## 1. Tensor

`torch.Tensor` is the central class of the package. Setting the attribute `.requires_grad` to `True` will make the tensor "record" all operations on it. When you finish your computation you can call `.backward()` and have all the gradients computed automatically. The gradient for this tensor will be accumulated into the `.grad` attribute.

![autograd.Variable](../static_files/autograd-variable.png)

There’s one more class which is very important for autograd implementation - a `Function`.

`Tensor` and `Function` are interconnected and build up an acyclic graph, that encodes a complete history of computation. Each tensor has a `.grad_fn` attribute that references a `Function` that has created the `Tensor` (except for Tensors created by the user - their `grad_fn` is `None`).

If you want to compute the derivatives, you can call `.backward()` on a Tensor. If `Tensor` is a scalar (i.e. it holds a one element data), you don’t need to specify any arguments to backward(), however if it has more elements, you need to specify a `gradient` argument that is a tensor of matching shape.

In [21]:
import torch

Create a tensor

In [22]:
x = torch.ones(2, 2, requires_grad=True)
print(x)

tensor([[1., 1.],
        [1., 1.]], requires_grad=True)


Do a tensor operation:

In [23]:
y = x + 2
print(y)

tensor([[3., 3.],
        [3., 3.]], grad_fn=<AddBackward0>)


`y` was created as a result of an operation, so it has a `grad_fn`.

In [24]:
print(y.grad_fn)

Do more operations on y

In [25]:
z = y * y * 3
out = z.mean()

print(z)
print(out)

tensor([[27., 27.],
        [27., 27.]], grad_fn=<MulBackward0>)
tensor(27., grad_fn=<MeanBackward0>)


# Assignments

1. Create a Tensor that `requires_grad` of size (5, 5)
2. Sum the values in the Tensor

In [26]:
# 1
x1 = torch.ones(5,5, requires_grad = True)
print(x1)

# 2
sum = x1.sum()

print(sum)

tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]], requires_grad=True)
tensor(25., grad_fn=<SumBackward0>)


## 2. Gradients

Let’s backprop now. Because `out` contains a single scalar, `out.backward()` is equivalent to `out.backward(torch.tensor([1.0]))`

In [27]:
out.backward()

Print gradients d(out)/dx

In [28]:
print(x.grad)

tensor([[4.5000, 4.5000],
        [4.5000, 4.5000]])


you can see,x.grad = 4.5
bc:
1. y = x + 2 and z = y * y * 3, out = z.mean()
2. so, dout / dz *dz/dy * dy / dx = dout / dx = 1/4 * 6y * 1
3. so, when y euqals 3, dout/dx = 4.5

You should have a matrix of `4.5`. Let’s denote the tensor `out` with $o$.

We have:
$o = \frac{1}{4}\sum_i z_i$, $z_i = 3(x_i+2)^2$ and $z_i\bigr\rvert_{x_i=1} = 27$.

Therefore, $\frac{\partial o}{\partial x_i} = \frac{3}{2}(x_i+2)$,
hence $\frac{\partial o}{\partial x_i}\bigr\rvert_{x_i=1} = \frac{9}{2} = 4.5$.

You can do many crazy things with autograd!

In [29]:
x = torch.randn(3, requires_grad=True)

y = x * 2
while y.data.norm() < 1000:
    y = y * 2

print(y)

tensor([-466.0936, 1422.0046,  215.9508], grad_fn=<MulBackward0>)


In [30]:
gradients = torch.FloatTensor([0.1, 1.0, 0.0001])
y.backward(gradients)

print(x.grad)

tensor([5.1200e+01, 5.1200e+02, 5.1200e-02])


核心就一句：**反向传播一次只能对「一个数」求导。** `y` 有三个数，系统不知道你要的是哪一种「导数」，所以必须你再补一个向量，把三个数合成一个问题。

---

## `backward` 在问什么

`x.grad` 的每个分量是：

\[
\frac{\partial \,(\text{某一个标量})}{\partial x_i}
\]

分子必须是 **一个** 数。`out.mean()`、`loss` 都是一个数，所以 `out.backward()` 问得很清楚：\(\partial\mathrm{out}/\partial x\)。

`y` 是 \((y_0,y_1,y_2)\)。下面这些都是不同的问题：

- 只要 \(\partial y_0/\partial x\) ？
- 只要 \(\partial y_1/\partial x\) ？
- 要 \(\partial(y_0+y_1+y_2)/\partial x\) ？
- 要 \(\partial(0.1 y_0 + 1\cdot y_1 + \cdots)/\partial x\) ？

不指定的话，数学上对应的是 **雅可比矩阵**（3 个输出 × 3 个输入），不是和 `x` 一样长的一个 `grad`。PyTorch 的 `.grad` 只给叶子准备了 **和 `x` 同形状的一块存储**，装不下整张矩阵，也没有默认选哪一行。

所以光写 `y.backward()`：问题没定义完，直接报错。

---

## 那个同形状向量在干什么

你传入 `v = [0.1, 1.0, 0.0001]`，等于先定义标量

\[
L = v_0 y_0 + v_1 y_1 + v_2 y_2
\]

现在分子又是一个数了，`backward` 可以跑。链式法则里这正好是：把 `v` 当作已经传到 `y` 上的 \(\partial L/\partial y\)（和 Nanograd 里的 `bp` 一样），再往 `x` 传。

- `v` 全是 1：等价于对 `y.sum()` 求导  
- `v` 是 `[1,0,0]`：只对 \(y_0\) 求导  
- 标量的 `out.backward()`：等价于 `v=1`，所以可以省略

不是 PyTorch 刁难你，是 **向量没有唯一的「对 x 的梯度」**；你必须用 `v` 说明「三个输出怎么加权合成一个 \(L\)」。

---

**一句话：** `y.backward()` 缺的是「对哪个标量求导」。同形状的向量就是在指定这个标量 \(L=v\cdot y\)，也就是外面传来的 \(\partial L/\partial y\)。训练时 loss 已经是标量，所以从来不用写这个参数。

**Read later** \
*Documentation* \
`Tensor`: https://pytorch.org/docs/stable/tensors.html \
`Function`: http://pytorch.org/docs/autograd

# Assignments

1. Define a tensor and set `requires_grad` to `True`
3. Multiply the tensor by 2 and assign the result to a new python variable (i.e. `x = result`)
4. Sum the variable's elements and assign to a new python variable
5. Print the gradients of all the variables
6. Now perform a backward pass on the last variable (NOTE: for each new python variable that you define, call `.retain_grad()`)
7. Print all gradients again
  - what did you notice?

In [37]:
a = torch.ones(6,6, requires_grad = True)

b = a * 2

c = b.sum()

print(c.grad)
print(b.grad)
print(a.grad)

c.backward()
print(a.grad)
print(b.grad)
print(c.grad)


print("=========================")

print("you can see, their grads are None except a, after c.backward(). because b and c are not leaf nodes, cpmputer want to save memory")

print("but you can use .retain_grad() to keep them")


print("=========================")
a = torch.ones(6,6, requires_grad = True)

b = a * 2

b.retain_grad()

c = b.sum()

print(c.grad)
print(b.grad)
print(a.grad)

c.backward()
print(a.grad)
print(b.grad)
print(c.grad)





None
None
None
tensor([[2., 2., 2., 2., 2., 2.],
        [2., 2., 2., 2., 2., 2.],
        [2., 2., 2., 2., 2., 2.],
        [2., 2., 2., 2., 2., 2.],
        [2., 2., 2., 2., 2., 2.],
        [2., 2., 2., 2., 2., 2.]])
None
None
you can see, their grads are None except a, after c.backward(). because b and c are not leaf nodes, cpmputer want to save memory
but you can use .retain_grad() to keep them
None
None
None
tensor([[2., 2., 2., 2., 2., 2.],
        [2., 2., 2., 2., 2., 2.],
        [2., 2., 2., 2., 2., 2.],
        [2., 2., 2., 2., 2., 2.],
        [2., 2., 2., 2., 2., 2.],
        [2., 2., 2., 2., 2., 2.]])
tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])
None


/var/folders/jc/skvq2k7x7xj85dwnw9bx72380000gn/T/ipykernel_33762/3466569445.py:7: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /Users/runner/work/pytorch/pytorch/build/aten/src/ATen/core/TensorBody.h:493.)
  print(c.grad)
/var/folders/jc/skvq2k7x7xj85dwnw9bx72380000gn/T/ipykernel_33762/3466569445.py:8: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you acces